### Ingestion del archivo "movie_genre.json"

In [0]:
%run "../../utilerias/funciones_comunes"

In [0]:
%run "../../utilerias/configurationes"

#### Paso 1 - Leer el archivo "movie_genre.json" usando "DataFrameReader" de spark

In [0]:
dbutils.widgets.text("param_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("param_file_date")


In [0]:
dbutils.widgets.text("parm_enviroment","PRODUCTION")
v_enviroment = dbutils.widgets.get("parm_enviroment")

In [0]:
#Importamos tipos de datos
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
#Definimos esquemas
schema_movie_genre = StructType([
    StructField('movieId', IntegerType(), False),
    StructField('genreId', IntegerType(), False)
]) 

In [0]:
#Leer los datos
movie_genre_df = spark.read.option("header", True).schema(schema_movie_genre).json(f"{raw_folder_path}/{v_file_date}/movie_genre.json")


#### Paso 2 - Añadir nuevas columnas
 1. Agrega las columnas "ingestion_date" y "enviroment" 

In [0]:
#Importaciones
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
movie_genre_df_final = movie_genre_df.withColumn("ingestion_date", current_timestamp())\
.withColumn("enviroment", lit(v_enviroment))\
.withColumn("file_date", lit(v_file_date))

#### Paso 3 - Escribir la salida en un formato "Delta Table"[](url) con PartitionBy basado en movieId

In [0]:
merge_condition = 'tgt.movieId = src.movieId AND tgt.genreId = src.genreId  AND tgt.file_date = src.file_date'
merge_delta_lake("smartdata_proyect_bronze","movies_genres",movie_genre_df_final, merge_condition, "file_date")

In [0]:
%sql
select count(1), file_date from smartdata_proyect_bronze.movies_genres group by file_date;

In [0]:
dbutils.notebook.exit("success")